In [ ]:
# =========================
# JSON → Power BI (Easy/Flat) NOTES
# =========================
# Goal: Turn API-style JSON into a clean table (CSV) for Power BI.
#
# Standard DE steps:
# 1) Confirm file path exists (avoid 50% of beginner errors).
# 2) Load JSON (json.load / json.loads).
# 3) Inspect the "shape":
#    - Top-level keys? Usually {"meta": ..., "data": [...]}
#    - Where are the rows? Usually payload["data"] (list of dicts)
#    - Check 1 sample record to know columns
# 4) Convert to table:
#    - df = pd.DataFrame(payload["data"])
# 5) Clean + standardize types:
#    - timestamps: pd.to_datetime(..., errors="coerce", utc=True)
#    - numbers: pd.to_numeric(..., errors="coerce")
#    - missing values: fillna(...) if needed
# 6) Quality checks (quick but important):
#    - duplicates: df["id"].duplicated().sum()
#    - missing counts: df.isna().sum()
#    - sanity rules: amounts >= 0, timestamps not null
# 7) Export:
#    - df.to_csv("file.csv", index=False)
#
# Reminder: dict has no numeric index like list.
# - dict uses keys: d["key"]
# - list uses positions: lst[0]
#
# For nested JSON (later levels):
# - Flatten dicts into columns (e.g., user.id → user_id)
# - Explode lists into rows (e.g., items[] → one row per item)
# =========================


#### Import Library

In [ ]:
# NOTE: json = read JSON
# NOTE: Path = safer file paths than raw strings
# NOTE: pandas = convert to table + export CSV

import json
from pathlib import Path
import pandas as pdm


#### File path + existence check

In [4]:
# NOTE: Put the JSON file in the same folder as your notebook,
# or change the path here.

input_path = Path("easy_orders.json")
input_path

WindowsPath('easy_orders.json')

In [6]:
# NOTE: Always check the file exists before reading, saves time.
print("Exists?", input_path.exists)
print("Absolute path:", input_path.resolve())

Exists? <bound method Path.exists of WindowsPath('easy_orders.json')>
Absolute path: C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\easy_orders.json


#### Load JSON + inspect the “shape”

In [16]:
# NOTE: Read the file text then parse JSON into Python objects (dict/list).
payload = json.loads(input_path.read_text(encoding="utf-8"))
payload

{'meta': {'source': 'demo_api', 'pulled_at': '2026-02-05T00:00:00Z'},
 'data': [{'order_id': 'ord_001',
   'order_time': '2026-02-04T10:12:30Z',
   'customer_id': 101,
   'city': 'Sydney',
   'total_amount': 34.5,
   'currency': 'AUD'},
  {'order_id': 'ord_002',
   'order_time': '2026-02-04T10:15:05Z',
   'customer_id': 102,
   'city': 'Melbourne',
   'total_amount': 12.0,
   'currency': 'AUD'},
  {'order_id': 'ord_003',
   'order_time': '2026-02-04T11:02:11Z',
   'customer_id': 101,
   'city': 'Sydney',
   'total_amount': 7.25,
   'currency': 'AUD'}]}

In [21]:

# NOTE: Confirm top level structures
print("Top-level keys:", list(payload.keys()))

# NOTE: meta is usually infor about the API pull; not rows.
print("Meta: ", payload.get("meta"))

# NOTE: data is where the rows usually are (list of dict records).
data = payload.get('data',[])
print("Type of data", type(data))
print("Number of records", len(data))

# NOTE: Always print one record to understand columns and types.
print("First Records", data[0] if data else None)

Top-level keys: ['meta', 'data']
Meta:  {'source': 'demo_api', 'pulled_at': '2026-02-05T00:00:00Z'}
Type of data <class 'list'>
Number of records 3
First Records {'order_id': 'ord_001', 'order_time': '2026-02-04T10:12:30Z', 'customer_id': 101, 'city': 'Sydney', 'total_amount': 34.5, 'currency': 'AUD'}


#### Convert to DataFrame (table)

In [24]:
# NOTE: A list of dicts is the easiest case: directly become a talbe.
df = pd.DataFrame(data)
df

,order_id,order_time,customer_id,city,total_amount,currency
0,ord_001,2026-02-04T10:12:30Z,101,Sydney,34.50,AUD
1,ord_002,2026-02-04T10:15:05Z,102,Melbourne,12.00,AUD
2,ord_003,2026-02-04T11:02:11Z,101,Sydney,7.25,AUD


#### Clean data types

In [27]:
# NOTE: APIs often return timestamps as strings. We convert to dateime.
# NOTE: errors="coerce" means bad timestamps become Nat(null) instead of crashing.

df["order_time"] = pd.to_datetime(df["order_time"], utc= True, errors="coerce")

# NOTE: Convert numeric fields safely. Strings like "101" become 101.
# NOTE: errors="coerce" turns bad values into NaN (null).
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")
df["total_amount"] = pd.to_numeric(df["total_amount"], errors="coerce")

# NOTE: Quick check after cleaning.
print("Dtypes: ")
display(df.dtypes)

# NOTE: Print missing values.
print("Missing values per columns: ")
display(df.isna().sum())

df



Dtypes: 


order_id                        str
order_time      datetime64[us, UTC]
customer_id                   int64
city                            str
total_amount                float64
currency                        str
dtype: object

Missing values per columns: 


order_id        0
order_time      0
customer_id     0
city            0
total_amount    0
currency        0
dtype: int64

,order_id,order_time,customer_id,city,total_amount,currency
0,ord_001,2026-02-04 10:12:30+00:00,101,Sydney,34.50,AUD
1,ord_002,2026-02-04 10:15:05+00:00,102,Melbourne,12.00,AUD
2,ord_003,2026-02-04 11:02:11+00:00,101,Sydney,7.25,AUD


#### Data Quality checks 

In [33]:
# NOTE: order_id should be unique in a clean orders table.
dup_orders = df["order_id"].duplicated().sum()
print("Duplicate order_id count", dup_orders)

# NOTE: timestamps should parse successfully (NaT means bad format).
bad_time = df["order_time"].isna().sum()
print("Bad timestamps (NaT) count:", bad_time)

# NOTE: Sanity check: amounts shouldn't be negative,
neg_amount = (df["total_amount"] < 0).sum()
print("Negative total_amount rows:", neg_amount)

# NOTE: Quick BI-style aggragagion: total sales by city.
sales_by_city = df.groupby("city", as_index=False)["total_amount"].sum()
sales_by_city

Duplicate order_id count 0
Bad timestamps (NaT) count: 0
Negative total_amount rows: 0


,city,total_amount
0,Melbourne,12.00
1,Sydney,41.75


#### Add a Power BI- friendly date column

In [35]:
# NOTE: Power BI time visuals are easier with a date column.
df["order_date"] = df["order_time"].dt.date

df.head()

,order_id,order_time,customer_id,city,total_amount,currency,order_date
0,ord_001,2026-02-04 10:12:30+00:00,101,Sydney,34.50,AUD,2026-02-04
1,ord_002,2026-02-04 10:15:05+00:00,102,Melbourne,12.00,AUD,2026-02-04
2,ord_003,2026-02-04 11:02:11+00:00,101,Sydney,7.25,AUD,2026-02-04


#### Export to SAV (Power BI Input)

In [36]:
# NOTE: CSV is the simplese import format for POWER BI.
output_path = Path("easy_order_powerbi.csV")
df.to_csv(output_path, index=False, encoding="utf-8")

print("Saved CSV:", output_path.resolve())

Saved CSV: C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\easy_order_powerbi.csV
